# Diplomado en Ciencia de Datos UC

### Módulo: Modelos Matemáticos y Estadísticos Clásicos
### Curso: Modelos ANOVA y Regresión Lineal
### Sesión 19: Regresión Lineal Múltiple
### Profesora: Ana María Alvarado
#### Fecha: 09 de Julio de 2026

In [ ]:
install.packages("janitor")
install.packages("nortest")
install.packages("car")
install.packages("lmtest")
install.packages("corrplot")



In [ ]:
library(tidyverse)

# 1. Regresión Lineal Múltiple

¿Se puede determinar la biomasa en función de el ph y buf?

In [ ]:
df_bio <- read.delim("biomass.txt")|>
  janitor::clean_names()
head(df_bio)

In [ ]:
# Modelo base

lm_bio0 = lm(bio ~ 1 , data = df_bio)
summary(lm_bio0)



In [ ]:
# Modelo de Regresión Simple

lm_bio1 = lm(bio ~ ph , data = df_bio)
summary(lm_bio1)


In [ ]:
## Modelo de Regresión Lineal Múltipla
lm_bio2 = lm(bio ~ ph + buf, data = df_bio)
summary(lm_bio2)


In [ ]:
lm_bio3 = lm(bio ~ ph + buf + sal, data = df_bio)
summary(lm_bio3)

# 2. Modelo de Regresión Múltiple con Variables Cualitativas

In [ ]:
df_bio |>
  ggplot( aes(x = ph, y = bio, color = type)) +
  geom_point(size = 3) +
  labs(title = "Biomasa aérea vs. Salinidad del suelo",
       x = "PH del suelo en agua (PH)", y = "Biomasa aérea (BIO)") +
  theme_bw(base_size = 15)


In [ ]:
lm_bio4 <- lm(bio ~ ph + type, data = df_bio)
summary(lm_bio4)


In [ ]:
head(model.matrix(lm_bio4))

In [ ]:
lm_bio5 <- lm(bio ~ ph + sal + type, data = df_bio)
summary(lm_bio5)

# 3. Selección de Variables

In [ ]:
lm_nulo     <- lm(bio ~ 1, data = df_bio)
lm_completo <- lm(bio ~ ., data = df_bio)

## Forward

In [ ]:
lm_fw <- step(lm_nulo,   scope =  formula(lm_completo), direction = "forward")
lm_fw

In [ ]:
summary(lm_fw)

## Backward

In [ ]:
lm_bc <- step(lm_completo, direction = "backward")
lm_bc

In [ ]:
summary(lm_bc)

## Both

In [ ]:
lm_bt <- step(lm_completo, direction = "both")
lm_bt

In [ ]:
summary(lm_bt)

In [ ]:
AIC(lm_completo)
AIC(lm_fw)
AIC(lm_bc)
AIC(lm_bt)

In [ ]:
data.frame(
  Modelo = c("Completo", "Forward", "Backward", "Bidirectional"),
  AIC = c(
    AIC(lm_completo),
    AIC(lm_fw),
    AIC(lm_bc),
    AIC(lm_bt)
  ),
  BIC = c(
    BIC(lm_completo),
    BIC(lm_fw),
    BIC(lm_bc),
    BIC(lm_bt)
  ),
  R2_Ajustado = c(
    summary(lm_completo)$adj.r.squared,
    summary(lm_fw)$adj.r.squared,
    summary(lm_bc)$adj.r.squared,
    summary(lm_bt)$adj.r.squared
  )
)

# 4. Multicolinealidad

In [ ]:
cor_bio <- df_bio |>
  select_if(is.numeric) |>
  cor()|>
  round(2)

  corrplot::corrplot(cor_bio, order = 'AOE', type = 'lower', addCoef.col = "black", number.cex = 0.5)

## Factor de Inflación de la Varianza (VIF)


In [ ]:
car::vif(lm_completo)


In [ ]:
car::vif(lm_bio3)

# 5. Análisis de Residuos




In [ ]:
residuos <- residuals(lm_fw)

In [ ]:
hist(residuos)

In [ ]:
plot(lm_fw)




## a. Normalidad


H0: Los datos siguen una distribución Normal.

H1: Los datos no siguen una distribución Normal.

In [ ]:
ggplot(data=NULL, aes(sample = residuos))+
  stat_qq() + stat_qq_line()


In [ ]:
shapiro.test(residuos)


## b. Independencia

H0: Los errores no están autocorrelacionados.

H1: Los errores presentan correlación a un paso.

In [ ]:

lmtest::dwtest(lm_fw)



## c. Homocedasticidad

H0: Los errores son homocedásticos

H1: Los errores no son heterocedásticos

In [ ]:
lmtest::bptest(lm_fw)

# 6. Valores atipicos y de alta influencia


In [ ]:
car::influencePlot(lm_fw)

# 6. Predicción

In [ ]:
df_bio_new <- read.csv2("bio_new.csv")|>
  janitor::clean_names()

head(df_bio_new)